# CBT RAG System - Knowledge Base + Conversation Examples

**Pipeline**: After M5 (LLM) generates response, RAG retrieves grounding context.

| RAG | Purpose | Files |
|---|---|---|
| RAG 1 - Knowledge | CBT facts, definitions, distortions | intents.json, counsel-chat.csv |
| RAG 2 - Examples | Real counselor-client exchanges | MentalChat16K.csv, Amod.csv |

**Stack**: BAAI/bge-m3 embedder + Qdrant local + BM25 hybrid retrieval

**Run order**: Setup -> RAG 1 -> RAG 2 -> BM25 -> Test -> Export zip


In [1]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Bước 1: Gỡ torchvision để tránh torch.library.register_fake conflict
# (giống các notebook training — không cần torchvision cho RAG)
import subprocess
subprocess.run(["pip", "uninstall", "-y", "torchvision"], capture_output=True)

# Bước 2: Pin protobuf==3.20.3 TRƯỚC KHI install qdrant-client
# protobuf >= 4.x bỏ GetPrototype → qdrant-client 1.x bị crash
!pip install -q "protobuf==3.20.3" 2>&1 | tail -1

# Bước 3: Install các thư viện RAG
!pip install -q sentence-transformers==3.2.0 qdrant-client==1.12.0 rank-bm25==0.2.2 2>&1 | tail -2

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
print(f"Device:  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Verify protobuf version
import google.protobuf
print(f"Protobuf: {google.protobuf.__version__}  (cần 3.20.x)")


grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
PyTorch: 2.10.0+cu128
CUDA:    True
Device:  Tesla T4
Protobuf: 6.33.6  (cần 3.20.x)


In [2]:
# ============================================================
# CONFIG - adjust paths to match your Kaggle dataset name
# ============================================================

import os, json, re, html, uuid, time
from pathlib import Path

# Kaggle input paths (update dataset name)
RAG1_INTENTS_PATH = "/kaggle/input/datasets/vipaccpes/mental-health/intents.json"
RAG1_COUNSEL_PATH = "/kaggle/input/datasets/vipaccpes/counsel-chat/counsel-chat.csv"
RAG2_MENTAL_PATH  = "/kaggle/input/datasets/vipaccpes/mentalchat16k/MentalChat16K.csv"
RAG2_AMOD_PATH    = "/kaggle/input/datasets/vipaccpes/amod-data/Amod.csv"

# Output paths
OUT_DIR    = "/kaggle/working/rag_outputs"
QDRANT_DIR = "/kaggle/working/qdrant_storage"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(QDRANT_DIR, exist_ok=True)

# Model + DB
EMBED_MODEL    = "BAAI/bge-m3"
EMBED_DIM      = 1024
RAG1_COLLECTION = "cbt_knowledge"
RAG2_COLLECTION = "cbt_examples"
MAX_CHUNK_CHARS = 1500
BATCH_SIZE     = 16    # giảm từ 64→16 tránh OOM

print("CONFIG ready")
print(f"  Embedder:  {EMBED_MODEL} ({EMBED_DIM}-dim)")
print(f"  Qdrant:    {QDRANT_DIR}")
print(f"  RAG 1:     {RAG1_COLLECTION}")
print(f"  RAG 2:     {RAG2_COLLECTION}")


CONFIG ready
  Embedder:  BAAI/bge-m3 (1024-dim)
  Qdrant:    /kaggle/working/qdrant_storage
  RAG 1:     cbt_knowledge
  RAG 2:     cbt_examples


In [3]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct, Filter, FieldCondition, MatchValue
import numpy as np
import pandas as pd

print(f"Loading {EMBED_MODEL}...")
# Dùng CPU cho embedder — nhỏ (~90MB), tránh OOM khi share GPU với LLM
embedder = SentenceTransformer(EMBED_MODEL, device="cpu")
print(f"  Embedder ready ({embedder.get_sentence_embedding_dimension()}-dim)")

qdrant = QdrantClient(path=QDRANT_DIR)

for col in [RAG1_COLLECTION, RAG2_COLLECTION]:
    try: qdrant.delete_collection(col)
    except: pass

qdrant.create_collection(
    collection_name=RAG1_COLLECTION,
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
)
qdrant.create_collection(
    collection_name=RAG2_COLLECTION,
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
)
print(f"  Created collections: {RAG1_COLLECTION}, {RAG2_COLLECTION}")


/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2026-05-13 01:29:48.009209: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778635788.193355      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778635788.245555      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778635788.689035      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the sam

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

  Embedder ready (1024-dim)
  Created collections: cbt_knowledge, cbt_examples


In [4]:
# RAG 1: CBT KNOWLEDGE BASE

def clean_text(t):
    if not isinstance(t, str): return ""
    t = html.unescape(t)
    t = re.sub(r"http\S+|www\.\S+", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

rag1_chunks = []

# Source 1: intents.json
print("[Source 1] intents.json")
if os.path.exists(RAG1_INTENTS_PATH):
    with open(RAG1_INTENTS_PATH) as f:
        intents_data = json.load(f)
    
    if isinstance(intents_data, dict) and "intents" in intents_data:
        intents_list = intents_data["intents"]
    elif isinstance(intents_data, list):
        intents_list = intents_data
    else:
        intents_list = []
    
    for intent in intents_list:
        tag       = intent.get("tag", "")
        patterns  = intent.get("patterns", [])
        responses = intent.get("responses", [])
        
        if responses:
            question_part = " | ".join(patterns[:3]) if patterns else tag
            answer_part   = " ".join(responses[:3])
            chunk_text = "Topic: " + tag + "\nQuestion examples: " + question_part + "\nAnswer: " + answer_part
            chunk_text = clean_text(chunk_text)[:MAX_CHUNK_CHARS]
            
            if len(chunk_text) >= 50:
                rag1_chunks.append({
                    "id":     str(uuid.uuid4()),
                    "text":   chunk_text,
                    "type":   "intent",
                    "source": "intents.json",
                    "metadata": {"tag": tag, "n_patterns": len(patterns), "n_responses": len(responses)},
                })
    cnt = sum(1 for c in rag1_chunks if c["source"] == "intents.json")
    print(f"  Built {cnt} intent chunks")
else:
    print(f"  Not found: {RAG1_INTENTS_PATH} - upload intents.json to continue")

# Source 2: counsel-chat.csv
print("\n[Source 2] counsel-chat.csv")
if os.path.exists(RAG1_COUNSEL_PATH):
    df_cc = pd.read_csv(RAG1_COUNSEL_PATH)
    print(f"  Loaded: {len(df_cc)} rows, columns: {df_cc.columns.tolist()}")
    
    for _, row in df_cc.iterrows():
        q_title = clean_text(str(row.get("questionTitle", "")))
        q_text  = clean_text(str(row.get("questionText", "")))
        topic   = clean_text(str(row.get("topic", "general")))
        answer  = clean_text(str(row.get("answerText", "")))
        upvotes = int(row.get("upvotes", 0)) if pd.notna(row.get("upvotes")) else 0
        
        if not answer or len(answer) < 50:
            continue
        
        chunk_text = "Topic: " + topic + "\nQ: " + q_title + ". " + q_text + "\nA: " + answer
        chunk_text = chunk_text[:MAX_CHUNK_CHARS]
        
        rag1_chunks.append({
            "id":     str(uuid.uuid4()),
            "text":   chunk_text,
            "type":   "qa_counsel",
            "source": "counsel-chat.csv",
            "metadata": {"topic": topic, "upvotes": upvotes, "q_length": len(q_text), "a_length": len(answer)},
        })
    cnt = sum(1 for c in rag1_chunks if c["source"] == "counsel-chat.csv")
    print(f"  Built {cnt} Q&A chunks")
else:
    print(f"  Not found: {RAG1_COUNSEL_PATH}")

print(f"\n  TOTAL RAG 1 chunks: {len(rag1_chunks)}")
for src in set(c["source"] for c in rag1_chunks):
    n = sum(1 for c in rag1_chunks if c["source"] == src)
    print(f"    {src}: {n}")

if rag1_chunks:
    s = rag1_chunks[0]
    print(f"\n  Sample chunk:")
    print(f"    type={s['type']}, source={s['source']}")
    print(f"    text: {s['text'][:250]}...")


[Source 1] intents.json
  Built 80 intent chunks

[Source 2] counsel-chat.csv
  Loaded: 2775 rows, columns: ['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views']
  Built 2744 Q&A chunks

  TOTAL RAG 1 chunks: 2824
    intents.json: 80
    counsel-chat.csv: 2744

  Sample chunk:
    type=intent, source=intents.json
    text: Topic: greeting Question examples: Hi | Hey | Is anyone there? Answer: Hello there. Tell me how are you feeling today? Hi there. What brings you here today? Hi there. How are you feeling today?...


In [ ]:
if not rag1_chunks:
    print('No RAG 1 chunks - upload data files first')
else:
    print(f'Embedding {len(rag1_chunks)} chunks...')
    start = time.time()
    texts = [c['text'] for c in rag1_chunks]
    embeddings_rag1 = embedder.encode(
        texts, batch_size=BATCH_SIZE, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    )
    elapsed = time.time() - start
    print(f'  Embedded in {elapsed:.1f}s ({len(rag1_chunks)/elapsed:.1f}/s)')
    print(f'  Shape: {embeddings_rag1.shape}')

    print(f'\n  Indexing to {RAG1_COLLECTION}...')
    n_indexed = 0
    for i in range(0, len(rag1_chunks), 256):
        batch_chunks = rag1_chunks[i:i+256]
        batch_embs   = embeddings_rag1[i:i+256]
        points = [
            PointStruct(
                id=c['id'],
                vector=emb.tolist(),
                # BUG 1 FIX: 'id' must be in payload so retrieve_hybrid
                # can match sparse BM25 hits against dense UUID hits.
                payload={
                    'id':     c['id'],      # <── ADDED
                    'text':   c['text'],
                    'type':   c['type'],
                    'source': c['source'],
                    **c['metadata'],
                },
            )
            for c, emb in zip(batch_chunks, batch_embs)
        ]
        qdrant.upsert(collection_name=RAG1_COLLECTION, points=points)
        n_indexed += len(points)
    print(f'  Indexed {n_indexed} points')

    rag1_jsonl = OUT_DIR + '/rag1_chunks.jsonl'
    with open(rag1_jsonl, 'w') as f:
        for c in rag1_chunks:
            f.write(json.dumps(c) + '\n')
    print(f'  Saved: {rag1_jsonl}')


In [6]:
# RAG 2: CONVERSATION EXAMPLES

rag2_chunks = []

# Source 1: MentalChat16K.csv
print("[Source 1] MentalChat16K.csv")
if os.path.exists(RAG2_MENTAL_PATH):
    df_mc = pd.read_csv(RAG2_MENTAL_PATH)
    print(f"  Loaded: {len(df_mc)} rows, columns: {df_mc.columns.tolist()}")
    
    input_col = None
    output_col = None
    for cand in ["input", "context", "instruction", "client", "user", "question", "Context"]:
        if cand in df_mc.columns:
            input_col = cand; break
    for cand in ["output", "response", "answer", "counselor", "assistant", "Response"]:
        if cand in df_mc.columns:
            output_col = cand; break
    
    if input_col and output_col:
        print(f"  Detected: input={input_col}, output={output_col}")
        for _, row in df_mc.iterrows():
            client_text = clean_text(str(row[input_col]))
            counselor   = clean_text(str(row[output_col]))
            if len(client_text) < 30 or len(counselor) < 50:
                continue
            
            chunk_text = "Client: " + client_text + "\nCounselor: " + counselor
            chunk_text = chunk_text[:MAX_CHUNK_CHARS]
            
            rag2_chunks.append({
                "id":              str(uuid.uuid4()),
                "text":            chunk_text,
                "type":            "conversation",
                "source":          "MentalChat16K.csv",
                "client_text":     client_text,
                "counselor_text":  counselor,
                "metadata":        {"client_len": len(client_text), "counselor_len": len(counselor)},
            })
        cnt = sum(1 for c in rag2_chunks if c["source"] == "MentalChat16K.csv")
        print(f"  Built {cnt} conversation chunks")
    else:
        print(f"  Cannot detect input/output columns")
else:
    print(f"  Not found: {RAG2_MENTAL_PATH}")

# Source 2: Amod.csv
print("\n[Source 2] Amod.csv")
if os.path.exists(RAG2_AMOD_PATH):
    df_am = pd.read_csv(RAG2_AMOD_PATH)
    print(f"  Loaded: {len(df_am)} rows, columns: {df_am.columns.tolist()}")
    
    for _, row in df_am.iterrows():
        client_text = clean_text(str(row.get("Context", "")))
        counselor   = clean_text(str(row.get("Response", "")))
        if len(client_text) < 30 or len(counselor) < 50:
            continue
        
        chunk_text = "Client: " + client_text + "\nCounselor: " + counselor
        chunk_text = chunk_text[:MAX_CHUNK_CHARS]
        
        rag2_chunks.append({
            "id":              str(uuid.uuid4()),
            "text":            chunk_text,
            "type":            "conversation",
            "source":          "Amod.csv",
            "client_text":     client_text,
            "counselor_text":  counselor,
            "metadata":        {"client_len": len(client_text), "counselor_len": len(counselor)},
        })
    cnt = sum(1 for c in rag2_chunks if c["source"] == "Amod.csv")
    print(f"  Built {cnt} conversation chunks")
else:
    print(f"  Not found: {RAG2_AMOD_PATH}")

print(f"\n  TOTAL RAG 2 chunks: {len(rag2_chunks)}")
for src in set(c["source"] for c in rag2_chunks):
    n = sum(1 for c in rag2_chunks if c["source"] == src)
    print(f"    {src}: {n}")

if rag2_chunks:
    s = rag2_chunks[0]
    print(f"\n  Sample (source={s['source']}):")
    print(f"    Client:    {s['client_text'][:150]}...")
    print(f"    Counselor: {s['counselor_text'][:200]}...")


[Source 1] MentalChat16K.csv
  Loaded: 16084 rows, columns: ['instruction', 'input', 'output']
  Detected: input=input, output=output
  Built 16044 conversation chunks

[Source 2] Amod.csv
  Loaded: 3512 rows, columns: ['Context', 'Response']
  Built 3493 conversation chunks

  TOTAL RAG 2 chunks: 19537
    Amod.csv: 3493
    MentalChat16K.csv: 16044

  Sample (source=MentalChat16K.csv):
    Client:    I've been struggling with my mental health for a while now, and I can't seem to find a way to cope with it. I've tried visualization, positive thinkin...
    Counselor: I understand that you've been dealing with a sense of confusion and chaos in your thoughts and emotions for some time now. It's been a challenging journey, and it's commendable that you've tried vario...


In [ ]:
# Giai phong GPU memory truoc khi embed
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(f'  GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

if not rag2_chunks:
    print('No RAG 2 chunks - upload MentalChat16K.csv + Amod.csv')
else:
    print(f'Embedding {len(rag2_chunks)} CLIENT TEXTS...')
    start = time.time()
    client_texts = [c['client_text'] for c in rag2_chunks]
    embeddings_rag2 = embedder.encode(
        client_texts, batch_size=BATCH_SIZE, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    )
    elapsed = time.time() - start
    print(f'  Embedded in {elapsed:.1f}s ({len(rag2_chunks)/elapsed:.1f}/s)')

    print(f'\n  Indexing to {RAG2_COLLECTION}...')
    n_indexed = 0
    for i in range(0, len(rag2_chunks), 256):
        batch_chunks = rag2_chunks[i:i+256]
        batch_embs   = embeddings_rag2[i:i+256]
        points = [
            PointStruct(
                id=c['id'],
                vector=emb.tolist(),
                # BUG 1 FIX: 'id' must be in payload
                payload={
                    'id':             c['id'],   # <── ADDED
                    'text':           c['text'],
                    'type':           c['type'],
                    'source':         c['source'],
                    'client_text':    c['client_text'],
                    'counselor_text': c['counselor_text'],
                    **c['metadata'],
                },
            )
            for c, emb in zip(batch_chunks, batch_embs)
        ]
        qdrant.upsert(collection_name=RAG2_COLLECTION, points=points)
        n_indexed += len(points)
    print(f'  Indexed {n_indexed} points')

    rag2_jsonl = OUT_DIR + '/rag2_chunks.jsonl'
    with open(rag2_jsonl, 'w') as f:
        for c in rag2_chunks:
            f.write(json.dumps(c) + '\n')
    print(f'  Saved: {rag2_jsonl}')


In [8]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text.lower())
    return [t for t in text.split() if len(t) > 2]

print("Building BM25 indexes...")

if rag1_chunks:
    rag1_tokenized = [tokenize(c["text"]) for c in rag1_chunks]
    bm25_rag1 = BM25Okapi(rag1_tokenized)
    print(f"  BM25 RAG 1: {len(rag1_chunks)} docs")
else:
    bm25_rag1 = None

if rag2_chunks:
    rag2_tokenized = [tokenize(c["client_text"]) for c in rag2_chunks]
    bm25_rag2 = BM25Okapi(rag2_tokenized)
    print(f"  BM25 RAG 2: {len(rag2_chunks)} docs")
else:
    bm25_rag2 = None


Building BM25 indexes...
  BM25 RAG 1: 2824 docs
  BM25 RAG 2: 19537 docs


In [ ]:
# UNIFIED RETRIEVAL FUNCTIONS

def retrieve_dense(query, collection, top_k=5, filter_source=None):
    q_emb = embedder.encode(query, normalize_embeddings=True, convert_to_numpy=True)
    query_filter = None
    if filter_source:
        query_filter = Filter(must=[FieldCondition(key='source', match=MatchValue(value=filter_source))])
    hits = qdrant.search(
        collection_name=collection,
        query_vector=q_emb.tolist(),
        limit=top_k,
        query_filter=query_filter,
    )
    return [{'score': h.score, 'payload': h.payload, 'id': h.id} for h in hits]


def retrieve_bm25(query, chunks, bm25_index, top_k=5):
    if bm25_index is None: return []
    scores = bm25_index.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [{'score': float(scores[i]), 'payload': chunks[i]} for i in top_idx]


# BUG 2 FIX: replace inconsistent mixed scoring with pure Reciprocal Rank Fusion.
# Old code mixed two incompatible approaches:
#   dense  -> alpha*normalized_score + (1-alpha)*RRF_bonus  (hybrid)
#   sparse -> (1-alpha)*normalized_score                     (normalised only)
# Pure RRF: score = 1/(k+rank) for every hit, regardless of source.
# Benefits: no extra hyperparameters, scale-invariant, well-studied.
def retrieve_hybrid(query, collection, chunks, bm25_index, top_k=5, alpha=0.7):
    # alpha retained for API compatibility but pure RRF does not use it
    dense_hits  = retrieve_dense(query, collection, top_k=top_k * 3)
    sparse_hits = retrieve_bm25(query, chunks, bm25_index, top_k=top_k * 3)

    combined = {}
    RRF_K = 60  # standard constant; higher -> smoother rank weighting

    # Dense pass
    for rank, h in enumerate(dense_hits):
        cid = h['id']
        combined[cid] = {
            'score':   1.0 / (RRF_K + rank),
            'payload': h['payload'],
        }

    # Sparse pass — correctly merges now because payload['id'] exists (Bug 1 fix)
    for rank, h in enumerate(sparse_hits):
        payload = h['payload']
        cid = payload.get('id', 'sparse_' + str(rank))
        if cid in combined:
            combined[cid]['score'] += 1.0 / (RRF_K + rank)  # merge scores
        else:
            combined[cid] = {
                'score': 1.0 / (RRF_K + rank),
                'payload': {
                    'id':             cid,
                    'text':           payload.get('text', ''),
                    'source':         payload.get('source', ''),
                    'client_text':    payload.get('client_text', ''),
                    'counselor_text': payload.get('counselor_text', ''),
                },
            }

    ranked = sorted(combined.values(), key=lambda x: x['score'], reverse=True)
    return ranked[:top_k]


def retrieve_for_inference(client_thought, top_k_knowledge=3, top_k_examples=3):
    knowledge = retrieve_hybrid(
        client_thought, RAG1_COLLECTION, rag1_chunks, bm25_rag1,
        top_k=top_k_knowledge,
    )
    examples = retrieve_hybrid(
        client_thought, RAG2_COLLECTION, rag2_chunks, bm25_rag2,
        top_k=top_k_examples,
    )
    return {'knowledge': knowledge, 'examples': examples}


print('Retrieval functions ready (pure RRF):')
print('  retrieve_dense(query, collection, top_k)')
print('  retrieve_bm25(query, chunks, bm25, top_k)')
print('  retrieve_hybrid(...)  <- pure RRF, no alpha dependency')
print('  retrieve_for_inference(client_thought, top_k_knowledge, top_k_examples)')


In [10]:
test_queries = [
    "I keep thinking everyone hates me, I am such a failure",
    "I cannot sleep at night because I am so anxious about tomorrow",
    "I do not have energy to do anything, I just stay in bed all day",
]

for q in test_queries:
    sep = "=" * 70
    print("\n" + sep)
    print("QUERY: " + q)
    print(sep)
    
    results = retrieve_for_inference(q, top_k_knowledge=2, top_k_examples=2)
    
    print("\n[KNOWLEDGE - Top 2 from RAG 1]")
    for i, r in enumerate(results["knowledge"]):
        p = r["payload"]
        text = p.get("text", "")
        print(f"  {i+1}. [score={r['score']:.3f}] source={p.get('source', '?')}")
        print(f"     {text[:200]}{'...' if len(text) > 200 else ''}")
    
    print("\n[EXAMPLES - Top 2 from RAG 2]")
    for i, r in enumerate(results["examples"]):
        p = r["payload"]
        print(f"  {i+1}. [score={r['score']:.3f}] source={p.get('source', '?')}")
        client = p.get("client_text", "")
        counselor = p.get("counselor_text", "")
        print(f"     Client:    {client[:150]}{'...' if len(client) > 150 else ''}")
        print(f"     Counselor: {counselor[:200]}{'...' if len(counselor) > 200 else ''}")



QUERY: I keep thinking everyone hates me, I am such a failure

[KNOWLEDGE - Top 2 from RAG 1]
  1. [score=0.985] source=counsel-chat.csv
     Topic: self-esteem
Q: Why do I feel like everyone hates me?. I constantly feel like everyone is up against me and trying their best to shut me down. It's ruining my mood and even my whole self. I have...
  2. [score=0.939] source=counsel-chat.csv
     Topic: self-esteem
Q: Why am I so angry and jealous?. I'm in my late 20s, and I've never had a boyfriend or even been on a date. I have no friends. I hate Facebook because everyone else has kids and a...

[EXAMPLES - Top 2 from RAG 2]
  1. [score=0.607] source=Amod.csv
     Client:    I feel like I'm ugly, stupid, useless, and that I can't make anyone happy.
     Counselor: Check out my blog post on: Four-ways-add-self-esteem-friends-list/I hope this offers you some nuggets of helpfulness!
  2. [score=0.607] source=Amod.csv
     Client:    I feel like I'm ugly, stupid, useless, and that I can't mak

In [ ]:
# SESSION MEMORY
# CBT relies on continuity across turns. A plain dict-in-memory is sufficient
# for a single session; no vector store needed.

from collections import Counter


def create_session():
    # Returns a fresh session state for a new CBT conversation.
    return {
        'turns':               [],           # sliding window of last 5 turns
        'distortion_history':  Counter(),    # total distortions seen this session
        'emotion_trajectory':  [],           # emotions in chronological order
        'topics_covered':      set(),        # topics discussed so far
    }


def update_session(session, client_text, counselor_text,
                   distortions=None, emotion=None, topic=None):
    # Call AFTER generating each counselor response to keep session current.
    session['turns'].append({
        'client':     client_text,
        'counselor':  counselor_text,
        'distortions': distortions or [],
        'emotion':    emotion or '',
    })
    session['turns'] = session['turns'][-5:]   # keep last 5 turns only
    if distortions:
        session['distortion_history'].update(distortions)
    if emotion:
        session['emotion_trajectory'].append(emotion)
    if topic:
        session['topics_covered'].add(topic)
    return session


def format_session_context(session):
    # Format session history as a string for injection into build_augmented_prompt.
    if not session['turns']:
        return ''
    lines = ['[SESSION CONTEXT]']
    if session['emotion_trajectory']:
        lines.append('Emotion trajectory: ' + ' -> '.join(session['emotion_trajectory'][-5:]))
    if session['distortion_history']:
        top_dist = session['distortion_history'].most_common(3)
        lines.append('Recurring distortions: ' + ', '.join(f'{d}(x{n})' for d, n in top_dist))
    if session['topics_covered']:
        lines.append('Topics covered: ' + ', '.join(sorted(session['topics_covered'])))
    lines.append('\nRecent turns:')
    for i, t in enumerate(session['turns'][-3:], 1):
        lines.append(f'  Turn {i}:')
        lines.append(f"    Client:    {t['client'][:200]}")
        lines.append(f"    Counselor: {t['counselor'][:200]}")
    return '\n'.join(lines)


print('Session memory functions ready:')
print('  create_session()                              -> fresh session dict')
print('  update_session(session, client, counselor)    -> call after each turn')
print('  format_session_context(session)               -> string for prompt injection')


In [ ]:
# PROMPT AUGMENTATION FOR M5 LLM INTEGRATION
# Improvements:
#   1. Token budget  — drop lowest-priority content before hitting context limit
#   2. VALID_ATTITUDES — replaces freeform string; warns on unknown values
#   3. Session context — [SESSION CONTEXT] prepended when session is provided

VALID_ATTITUDES = {'empathetic', 'collaborative', 'gentle-challenging',
                   'supportive', 'validating'}

# Qwen2.5-7B has 32K context but attention quality degrades after ~8K tokens.
# Keep the RAG context portion under ~6K tokens (rough: 1 token ~ 4 chars).
MAX_PROMPT_TOKENS = 6000

def _approx_tokens(text):
    return len(text) // 4


def build_augmented_prompt(client_thought, intake_form, emotion, severity,
                            distortions, attitude='empathetic',
                            top_k_knowledge=2, top_k_examples=2,
                            session=None):

    # Validate attitude
    if attitude not in VALID_ATTITUDES:
        print(f"  [Warning] attitude='{attitude}' not in {VALID_ATTITUDES}; "
              "defaulting to 'empathetic'")
        attitude = 'empathetic'

    rag = retrieve_for_inference(client_thought,
                                 top_k_knowledge=top_k_knowledge,
                                 top_k_examples=top_k_examples)

    knowledge_str = ''
    if rag['knowledge']:
        knowledge_str = '\n[RELEVANT CBT KNOWLEDGE]\n'
        for i, r in enumerate(rag['knowledge']):
            text = r['payload'].get('text', '')[:400]
            knowledge_str += f'({i+1}) {text}\n'

    examples_str = ''
    if rag['examples']:
        examples_str = '\n[SIMILAR PAST CASES]\n'
        for i, r in enumerate(rag['examples']):
            client   = r['payload'].get('client_text', '')[:250]
            counselor = r['payload'].get('counselor_text', '')[:300]
            examples_str += f'({i+1}) Past client: {client}\n'
            examples_str += f'    Counselor response: {counselor}\n'

    distortions_str = ', '.join(distortions) if distortions else 'none detected'

    # Session context is highest priority — prepend to prompt
    session_str = ''
    if session:
        session_str = format_session_context(session) + '\n\n'

    # Assemble helper — examples can be dropped if over token budget
    def _assemble(include_examples):
        p  = session_str
        p += f'[CLIENT PROFILE]\n{intake_form[:500]}\n\n'
        p += f'[CURRENT THOUGHT]\n{client_thought}\n\n'
        p += '[PSYCHOLOGICAL ANALYSIS]\n'
        p += f'- Emotion: {emotion}\n'
        p += f'- Severity: {severity}\n'
        p += f'- Cognitive distortions: {distortions_str}\n'
        p += f'- Recommended tone: {attitude}\n'
        p += knowledge_str
        if include_examples:
            p += examples_str
        p += '\n[TASK]\n'
        p += '1. Use the CBT knowledge above to choose the most appropriate technique\n'
        if include_examples:
            p += '2. Reference patterns from similar past cases when relevant\n'
        p += f'3. Respond with a {attitude} tone, considering emotion ({emotion}) and severity ({severity})\n'
        return p

    prompt = _assemble(include_examples=True)

    # Token budget guard — drop examples first (lowest RAG priority)
    if _approx_tokens(prompt) > MAX_PROMPT_TOKENS:
        print(f'  [Token budget] ~{_approx_tokens(prompt)} tokens; dropping examples to fit window.')
        prompt = _assemble(include_examples=False)

    return prompt


print('=' * 70)
print('AUGMENTED PROMPT EXAMPLE (with session memory + token budget)')
print('=' * 70)

# Simulate a one-turn session
demo_session = create_session()
demo_session = update_session(
    demo_session,
    client_text="I feel like nobody at work respects me.",
    counselor_text="It sounds like you're feeling undervalued. Let's explore what's driving that thought.",
    distortions=['mind-reading'],
    emotion='frustration',
    topic='work',
)

demo_prompt = build_augmented_prompt(
    client_thought='I am going to fail this exam, my whole life will be ruined',
    intake_form='Name: Sarah, Age: 21, Student',
    emotion='fear',
    severity='Anxiety',
    distortions=['catastrophizing', 'fortune-telling'],
    attitude='gentle-challenging',   # valid value — no warning
    session=demo_session,
)
print(demo_prompt)
print(f'\n[Token estimate: ~{_approx_tokens(demo_prompt)} tokens]')


In [ ]:
import zipfile, pickle

print('Final collection stats:')
for col in [RAG1_COLLECTION, RAG2_COLLECTION]:
    info = qdrant.get_collection(col)
    print(f'  {col}: {info.points_count} points')

# BUG 3 FIX: Serialize BM25 indexes with pickle.
# Original code wrote a dense-only rag_interface.py, silently losing all BM25
# work at deployment. Fix: pickle both BM25 objects + their chunk lists.
print('\nSerializing BM25 indexes...')

bm25_rag1_path = OUT_DIR + '/bm25_rag1.pkl'
bm25_rag2_path = OUT_DIR + '/bm25_rag2.pkl'

if bm25_rag1:
    with open(bm25_rag1_path, 'wb') as f:
        pickle.dump({'bm25': bm25_rag1, 'chunks': rag1_chunks}, f)
    print(f'  Saved: {bm25_rag1_path}')
if bm25_rag2:
    with open(bm25_rag2_path, 'wb') as f:
        pickle.dump({'bm25': bm25_rag2, 'chunks': rag2_chunks}, f)
    print(f'  Saved: {bm25_rag2_path}')

# Write updated rag_interface.py with full hybrid retrieval + session memory
INTERFACE_CODE = (
    '# CBT RAG Inference Interface\\n# Full hybrid retrieval: dense + BM25 + RRF fusion\\n# Usage:\\n#   from rag_interface import build_augmented_prompt, create_session, update_session\\n#   session = create_session()\\n#   prompt  = build_augmented_prompt(..., session=session)\\n#   session = update_session(session, client_text, counselor_text, distortions, emotion)\\n\\nimport os, re, pickle\\nimport numpy as np\\nfrom collections import Counter\\nfrom qdrant_client import QdrantClient\\nfrom qdrant_client.models import Filter, FieldCondition, MatchValue\\nfrom sentence_transformers import SentenceTransformer\\nfrom rank_bm25 import BM25Okapi\\n\\nEMBED_MODEL      = \'BAAI/bge-m3\'\\nQDRANT_DIR       = \'./qdrant_storage\'\\nRAG1_COLLECTION  = \'cbt_knowledge\'\\nRAG2_COLLECTION  = \'cbt_examples\'\\nBM25_RAG1_PATH   = \'./rag_outputs/bm25_rag1.pkl\'\\nBM25_RAG2_PATH   = \'./rag_outputs/bm25_rag2.pkl\'\\nVALID_ATTITUDES  = {\'empathetic\', \'collaborative\', \'gentle-challenging\', \'supportive\', \'validating\'}\\nMAX_PROMPT_TOKENS = 6000\\nRRF_K            = 60\\n\\nembedder = SentenceTransformer(EMBED_MODEL, device=\'cpu\')\\nqdrant   = QdrantClient(path=QDRANT_DIR)\\n\\ndef _load_bm25(path):\\n    if not os.path.exists(path):\\n        print(f\'[Warning] BM25 not found: {path} -- dense-only fallback active\')\\n        return None, []\\n    with open(path, \'rb\') as f:\\n        data = pickle.load(f)\\n    return data[\'bm25\'], data[\'chunks\']\\n\\nbm25_rag1, rag1_chunks = _load_bm25(BM25_RAG1_PATH)\\nbm25_rag2, rag2_chunks = _load_bm25(BM25_RAG2_PATH)\\n\\ndef _tokenize(text):\\n    text = re.sub(r\'[^a-zA-Z0-9\\\\s]\', \' \', text.lower())\\n    return [t for t in text.split() if len(t) > 2]\\n\\ndef retrieve_dense(query, collection, top_k=5):\\n    q_emb = embedder.encode(query, normalize_embeddings=True, convert_to_numpy=True)\\n    hits  = qdrant.search(collection_name=collection, query_vector=q_emb.tolist(), limit=top_k)\\n    return [{\'score\': h.score, \'payload\': h.payload, \'id\': h.id} for h in hits]\\n\\ndef retrieve_bm25(query, chunks, bm25_index, top_k=5):\\n    if bm25_index is None: return []\\n    scores  = bm25_index.get_scores(_tokenize(query))\\n    top_idx = np.argsort(scores)[::-1][:top_k]\\n    return [{\'score\': float(scores[i]), \'payload\': chunks[i]} for i in top_idx]\\n\\ndef retrieve_hybrid(query, collection, chunks, bm25_index, top_k=5):\\n    dense_hits  = retrieve_dense(query, collection, top_k=top_k * 3)\\n    sparse_hits = retrieve_bm25(query, chunks, bm25_index, top_k=top_k * 3)\\n    combined    = {}\\n    for rank, h in enumerate(dense_hits):\\n        cid = h[\'id\']\\n        combined[cid] = {\'score\': 1.0 / (RRF_K + rank), \'payload\': h[\'payload\']}\\n    for rank, h in enumerate(sparse_hits):\\n        payload = h[\'payload\']\\n        cid     = payload.get(\'id\', \'sparse_\' + str(rank))\\n        if cid in combined:\\n            combined[cid][\'score\'] += 1.0 / (RRF_K + rank)\\n        else:\\n            combined[cid] = {\'score\': 1.0 / (RRF_K + rank), \'payload\': {\\n                \'id\': cid, \'text\': payload.get(\'text\',\'\'),\\n                \'source\': payload.get(\'source\',\'\'),\\n                \'client_text\': payload.get(\'client_text\',\'\'),\\n                \'counselor_text\': payload.get(\'counselor_text\',\'\'),\\n            }}\\n    return sorted(combined.values(), key=lambda x: x[\'score\'], reverse=True)[:top_k]\\n\\ndef retrieve_for_inference(client_thought, top_k_knowledge=3, top_k_examples=3):\\n    return {\\n        \'knowledge\': retrieve_hybrid(client_thought, RAG1_COLLECTION, rag1_chunks, bm25_rag1, top_k=top_k_knowledge),\\n        \'examples\':  retrieve_hybrid(client_thought, RAG2_COLLECTION, rag2_chunks, bm25_rag2, top_k=top_k_examples),\\n    }\\n\\ndef create_session():\\n    return {\'turns\': [], \'distortion_history\': Counter(),\\n            \'emotion_trajectory\': [], \'topics_covered\': set()}\\n\\ndef update_session(session, client_text, counselor_text,\\n                   distortions=None, emotion=None, topic=None):\\n    session[\'turns\'].append({\'client\': client_text, \'counselor\': counselor_text,\\n                              \'distortions\': distortions or [], \'emotion\': emotion or \'\'})\\n    session[\'turns\'] = session[\'turns\'][-5:]\\n    if distortions: session[\'distortion_history\'].update(distortions)\\n    if emotion:     session[\'emotion_trajectory\'].append(emotion)\\n    if topic:       session[\'topics_covered\'].add(topic)\\n    return session\\n\\ndef _format_session_context(session):\\n    if not session[\'turns\']: return \'\'\\n    lines = [\'[SESSION CONTEXT]\']\\n    if session[\'emotion_trajectory\']:\\n        lines.append(\'Emotion trajectory: \' + \' -> \'.join(session[\'emotion_trajectory\'][-5:]))\\n    if session[\'distortion_history\']:\\n        top = session[\'distortion_history\'].most_common(3)\\n        lines.append(\'Recurring distortions: \' + \', \'.join(f\'{d}(x{n})\' for d, n in top))\\n    if session[\'topics_covered\']:\\n        lines.append(\'Topics covered: \' + \', \'.join(sorted(session[\'topics_covered\'])))\\n    lines.append(\'\\\\nRecent turns:\')\\n    for i, t in enumerate(session[\'turns\'][-3:], 1):\\n        lines.append(f\'  Turn {i}:\')\\n        lines.append(f\\"    Client:    {t[\'client\'][:200]}\\")\\n        lines.append(f\\"    Counselor: {t[\'counselor\'][:200]}\\")\\n    return \'\\\\n\'.join(lines)\\n\\ndef _approx_tokens(text): return len(text) // 4\\n\\ndef build_augmented_prompt(client_thought, intake_form, emotion, severity,\\n                            distortions, attitude=\'empathetic\',\\n                            top_k_knowledge=2, top_k_examples=2, session=None):\\n    if attitude not in VALID_ATTITUDES:\\n        print(f\\"[Warning] attitude=\'{attitude}\' not recognised; using \'empathetic\'\\")\\n        attitude = \'empathetic\'\\n    rag = retrieve_for_inference(client_thought,\\n                                  top_k_knowledge=top_k_knowledge,\\n                                  top_k_examples=top_k_examples)\\n    knowledge_str = \'\'\\n    if rag[\'knowledge\']:\\n        knowledge_str = \'\\\\n[RELEVANT CBT KNOWLEDGE]\\\\n\'\\n        for i, r in enumerate(rag[\'knowledge\']):\\n            knowledge_str += f\\"({i+1}) {r[\'payload\'].get(\'text\',\'\')[:400]}\\\\n\\"\\n    examples_str = \'\'\\n    if rag[\'examples\']:\\n        examples_str = \'\\\\n[SIMILAR PAST CASES]\\\\n\'\\n        for i, r in enumerate(rag[\'examples\']):\\n            ct = r[\'payload\'].get(\'client_text\',\'\')[:250]\\n            co = r[\'payload\'].get(\'counselor_text\',\'\')[:300]\\n            examples_str += f\'({i+1}) Past client: {ct}\\\\n    Counselor: {co}\\\\n\'\\n    distortions_str = \', \'.join(distortions) if distortions else \'none detected\'\\n    session_str = (_format_session_context(session) + \'\\\\n\\\\n\') if session else \'\'\\n    def _assemble(include_examples):\\n        p  = session_str\\n        p += f\'[CLIENT PROFILE]\\\\n{intake_form[:500]}\\\\n\\\\n\'\\n        p += f\'[CURRENT THOUGHT]\\\\n{client_thought}\\\\n\\\\n\'\\n        p += \'[PSYCHOLOGICAL ANALYSIS]\\\\n\'\\n        p += f\'- Emotion: {emotion}\\\\n- Severity: {severity}\\\\n\'\\n        p += f\'- Cognitive distortions: {distortions_str}\\\\n\'\\n        p += f\'- Recommended tone: {attitude}\\\\n\'\\n        p += knowledge_str\\n        if include_examples: p += examples_str\\n        p += \'\\\\n[TASK]\\\\n\'\\n        p += \'1. Use the CBT knowledge above to choose the most appropriate technique\\\\n\'\\n        if include_examples:\\n            p += \'2. Reference patterns from similar past cases when relevant\\\\n\'\\n        p += f\'3. Respond with a {attitude} tone, emotion ({emotion}), severity ({severity})\\\\n\'\\n        return p\\n    prompt = _assemble(include_examples=True)\\n    if _approx_tokens(prompt) > MAX_PROMPT_TOKENS:\\n        print(f\'[Token budget] Prompt too long; dropping examples.\')\\n        prompt = _assemble(include_examples=False)\\n    return prompt\\n'
)

interface_path = OUT_DIR + '/rag_interface.py'
with open(interface_path, 'w') as f:
    f.write(INTERFACE_CODE)
print(f'\n  Saved updated rag_interface.py: {interface_path}')

# Package everything into zip
ZIP_PATH = '/kaggle/working/cbt_rag_final.zip'
print(f'\nPackaging to {ZIP_PATH}...')

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for root, _, files in os.walk(QDRANT_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, os.path.dirname(QDRANT_DIR))
            zf.write(file_path, arcname)
    for fname in ['rag1_chunks.jsonl', 'rag2_chunks.jsonl', 'rag_interface.py',
                  'bm25_rag1.pkl', 'bm25_rag2.pkl']:
        fpath = OUT_DIR + '/' + fname
        if os.path.exists(fpath):
            zf.write(fpath, 'rag_outputs/' + fname)

zip_size = os.path.getsize(ZIP_PATH) / 1e6
print(f'  Saved: {ZIP_PATH} ({zip_size:.1f} MB)')

sep = '=' * 60
print('\n' + sep + '\nSUMMARY (FIXED)\n' + sep)
print(f'  RAG 1 (knowledge): {len(rag1_chunks)} chunks')
print(f'  RAG 2 (examples):  {len(rag2_chunks)} chunks')
print(f'\n  Fixes applied:')
print("    [Bug 1] payload includes 'id'   -> hybrid fusion now merges correctly")
print('    [Bug 2] pure RRF scoring        -> consistent dense + sparse weighting')
print('    [Bug 3] BM25 pickled + loaded   -> rag_interface.py uses full hybrid')
print(f'\n  New features:')
print('    [+] Session memory  (create_session / update_session)')
print('    [+] Token budget guard (~6K token limit, graceful truncation)')
print('    [+] VALID_ATTITUDES  (replaces freeform string)')
print(f'\n  Zip contents: qdrant_storage/ + rag_outputs/{{chunks, BM25 pkl, interface}}')
print('\n  App integration:')
print('    1. Download cbt_rag_final.zip -> unzip')
print('    2. pip install sentence-transformers qdrant-client rank-bm25')
print('    3. from rag_interface import build_augmented_prompt, create_session, update_session')
print('    4. Combine with M5 LLM (Qwen2.5-7B + LoRA)')
